# Galaxy Zoo

<div>
<img src="https://raw.githubusercontent.com/illinois-ipaml/MachineLearningForPhysics/main/img/Project_GalaxyZoo-galaxypic.png" width=500></img>
</div>

## <span style="color:Orange">Learning Objectives</span>

By the end of this project you should be able to:

* explain what the Sloan Digital Sky Survey is and why galaxy morphology matters for understanding galaxy formation and evolution;
* describe how Galaxy Zoo turns many volunteer classifications into a decision tree of vote fractions;
* load and interpret the Galaxy Zoo Kaggle labels, and connect each of the 37 numbers to a question and answer in the decision tree; and
* explore the label distributions and some example images, the starting point for training a convolutional neural network in Project 02.

## <span style="color:Orange">Overview</span>

Understanding how and why we are here is one of the fundamental questions for the ages. Part of the answer lies in the origins of galaxies, such as our own Milky Way. Yet questions remain about how the Milky Way, or any of the hundreds of billions of other galaxies in the observable Universe, formed and evolved. Galaxies come in all shapes, sizes and colours, from beautiful spirals to huge ellipticals. Their **morphology** (shape and structure) records their dynamical history, including mergers, bars, bulges and spiral arms.

The Sloan Digital Sky Survey (SDSS) [[4]](https://doi.org/10.1086/301513) has imaged a large fraction of the sky and measured spectra for about a million galaxies, far too many for professional astronomers to classify by eye. In 2007 the **Galaxy Zoo** project began asking members of the public to classify the SDSS galaxies online. The first project collected morphologies for nearly 900,000 galaxies [[3]](https://doi.org/10.1111/j.1365-2966.2010.17432.x). **Galaxy Zoo 2** then asked volunteers a more detailed series of questions, organized as a decision tree, and published more than 16 million classifications of 304,122 galaxies [[1]](https://doi.org/10.1093/mnras/stt1458).

In 2013–2014, the [Galaxy Challenge](https://www.kaggle.com/competitions/galaxy-zoo-the-galaxy-challenge) on Kaggle [[5]](https://www.kaggle.com/competitions/galaxy-zoo-the-galaxy-challenge) asked participants to predict the Galaxy Zoo 2 vote fractions directly from the images. The winning entry used a rotation-invariant convolutional neural network [[2]](https://doi.org/10.1093/mnras/stv632). In this project you will explore that dataset. In Project 02 you will train a CNN on it.

## <span style="color:Orange">Data Sources</span>

| File (course mirror) | Size | Contents |
|---|---|---|
| [`training_solutions_rev1.zip`](https://courses.physics.illinois.edu/phys498mlp/sp2025/data/projects/GalaxyZoo/training_solutions_rev1.zip) | 4.9 MB | **The labels:** `GalaxyID` + 37 vote fractions for each of the 61,578 training galaxies |
| [`images_training_rev1.zip`](https://courses.physics.illinois.edu/phys498mlp/sp2025/data/projects/GalaxyZoo/images_training_rev1.zip) | 830 MB | 61,578 training images, `<GalaxyID>.jpg`, 424 × 424 pixel RGB (SDSS *gri* colour composites) |
| [`images_test_rev1.zip`](https://courses.physics.illinois.edu/phys498mlp/sp2025/data/projects/GalaxyZoo/images_test_rev1.zip) | 1.1 GB | 79,975 test images. The competition's evaluation set; **no labels are provided** |
| [`all_zeros_benchmark.zip`](https://courses.physics.illinois.edu/phys498mlp/sp2025/data/projects/GalaxyZoo/all_zeros_benchmark.zip), [`all_ones_benchmark.zip`](https://courses.physics.illinois.edu/phys498mlp/sp2025/data/projects/GalaxyZoo/all_ones_benchmark.zip), [`central_pixel_benchmark.zip`](https://courses.physics.illinois.edu/phys498mlp/sp2025/data/projects/GalaxyZoo/central_pixel_benchmark.zip) | < 1 MB each | Example **submissions** (predictions for the test galaxies) from trivial baseline methods. They show the submission format; they are **not** labels |

The original data are on [Kaggle](https://www.kaggle.com/competitions/galaxy-zoo-the-galaxy-challenge/data); downloading them there requires a Kaggle account and accepting the competition rules.

**About the labels.** The 37 numbers for each galaxy are *not* independent class probabilities. They are the weighted fractions of Galaxy Zoo volunteers who gave each response, propagated down the decision tree. The answers to the first question (Class 1) sum to 1. The answers to each later question sum to the fraction of volunteers who reached that question. For example, Class 2 ("edge-on?") sums to Class 1.2 ("features or disk").

The cell below downloads the small label file (and, on first use, the 830 MB training-image archive) and defines a helper to display a galaxy image directly from the zip archive, with no need to extract 61,578 files.

In [ ]:
import io
import os
import urllib.request
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

BASE_URL = "https://courses.physics.illinois.edu/phys498mlp/sp2025/data/projects/GalaxyZoo/"

def fetch(filename):
    """Download a file from the course mirror once; later calls reuse the local copy."""
    if not os.path.exists(filename):
        urllib.request.urlretrieve(BASE_URL + filename, filename)
    return filename

labels = pd.read_csv(fetch("training_solutions_rev1.zip"))   # pandas reads the CSV inside the zip directly
print(labels.shape)

train_zip = zipfile.ZipFile(fetch("images_training_rev1.zip"))  # ~830 MB, downloaded once

def show_galaxy(galaxy_id, ax=None):
    img = Image.open(io.BytesIO(train_zip.read(f"images_training_rev1/{galaxy_id}.jpg")))
    ax = ax or plt.gca()
    ax.imshow(img)
    ax.set_title(str(galaxy_id))
    ax.axis("off")

labels.head()

## <span style="color:Orange">Questions</span>

Answer each question in a markdown cell directly below it (add code cells where needed). For the background questions (01 to 03), aim for one to two well-organized paragraphs, and **cite your sources**. The references at the end of this notebook are a good starting point. Your notebook should run from top to bottom without errors.

### <span style="color:LightGreen">Question 01</span>

What is the Sloan Digital Sky Survey (SDSS) [[4]](https://doi.org/10.1086/301513)? What telescope and instruments does it use, what does it measure (imaging in which filters, spectroscopy), and what motivated scientists to create such a survey? The Galaxy Zoo images are *gri* colour composites. What does that mean?

### <span style="color:LightGreen">Question 02</span>

Galaxy Zoo guides its volunteers through a decision tree of questions. Why is it important to be able to classify galaxies by morphology? What do the different shapes of galaxies (for example ellipticals, spirals, bars, bulges and mergers) tell us about how they formed and evolved? What information do galaxy colours provide?

According to Ref. [[2]](https://doi.org/10.1093/mnras/stv632) (Section 2), the organizers of the Kaggle challenge removed many of the over-represented red elliptical and blue spiral galaxies from the dataset. Why would they do that?

### <span style="color:LightGreen">Question 03</span>

After a careful reading of Ref. [[1]](https://doi.org/10.1093/mnras/stt1458), especially Fig. 1 and Table 2, describe the Galaxy Zoo 2 decision tree. What are its tasks (questions) and possible responses? Which tasks are asked of every galaxy, and which only follow particular earlier answers? Trace the path a volunteer would follow for (a) a smooth, round elliptical galaxy and (b) a face-on barred spiral with two arms.

### <span style="color:LightGreen">Question 04</span>

The label file `training_solutions_rev1.csv` has the following columns:

```
GalaxyID,
Class1.1, Class1.2, Class1.3,
Class2.1, Class2.2,
Class3.1, Class3.2,
Class4.1, Class4.2,
Class5.1, Class5.2, Class5.3, Class5.4,
Class6.1, Class6.2,
Class7.1, Class7.2, Class7.3,
Class8.1, Class8.2, Class8.3, Class8.4, Class8.5, Class8.6, Class8.7,
Class9.1, Class9.2, Class9.3,
Class10.1, Class10.2, Class10.3,
Class11.1, Class11.2, Class11.3, Class11.4, Class11.5, Class11.6
```

Make a table that labels each column with its question and response in the decision tree. For example, `Class1.1` = Task 01 "Is the galaxy simply smooth and rounded…?", response "smooth".

### <span style="color:LightGreen">Question 05</span>

Here is the label row for the first galaxy in the training set, `GalaxyID` 100008:

```
100008,
0.383147, 0.616853, 0.0,
0.0, 0.616853,
0.038452149, 0.578400851,
0.418397819, 0.198455181,
0.0, 0.104752126, 0.512100874, 0.0,
0.054453, 0.945547,
0.201462524, 0.181684476, 0.0,
0.0, 0.0272265, 0.0, 0.0272265, 0.0, 0.0, 0.0,
0.0, 0.0, 0.0,
0.279952491, 0.138445328, 0.0,
0.0, 0.092885571, 0.0, 0.0, 0.0, 0.325512248
```

1. Using the explanation of the labels in the Data Sources section, verify numerically, for the whole training set, that the responses to each question sum to the vote fraction of the response that leads to it (for example, Class 2 sums to Class 1.2, and Class 10 and Class 11 each sum to Class 4.1). Which questions sum to 1?
2. For galaxy 100008, what is the most popular answer to each question it was asked? Trace the most likely path through the decision tree. Note that comparing raw numbers *across* different questions is not meaningful. Why not?
3. Display the image of galaxy 100008 with `show_galaxy(100008)`. Do you agree with the volunteers?

### <span style="color:LightGreen">Question 06</span>

Explore the label distributions for the whole training set:

1. Plot histograms of the three Class 1 vote fractions (smooth, features/disk, star/artifact). Assign each galaxy to the top-level category with the largest Class 1 vote fraction. How many galaxies fall into each category? Which type is the majority?
2. Many galaxies are ambiguous. What fraction of galaxies have a clear majority (say, a vote fraction above 0.8) for "smooth" or for "features/disk"? Show a few example images from each group, and a few ambiguous ones.
3. Choose at least two deeper questions (for example, bar or no bar, or the number of spiral arms). Plot their vote-fraction distributions and comment on them.
4. Is the distribution of galaxy types in this dataset representative of galaxies in the Universe? Relate your answer to Question 02.

*Optional extension:* the Kaggle score was the root-mean-square error (RMSE) between predicted and true values over all 37 outputs. Split the labelled training set into two halves. Compute the RMSE you would get on one half by predicting the mean label vector of the other half for every galaxy. Then try a smarter non-image baseline of your own. These numbers are the baselines your CNN in Project 02 has to beat.

## <span style="color:Orange">References</span>

__[<span style="color:Red">1</span>]__ K. W. Willett *et al.*, "Galaxy Zoo 2: detailed morphological classifications for 304,122 galaxies from the Sloan Digital Sky Survey," *Mon. Not. R. Astron. Soc.* **435**, 2835 (2013). [doi:10.1093/mnras/stt1458](https://doi.org/10.1093/mnras/stt1458), [arXiv:1308.3496](https://arxiv.org/abs/1308.3496)

__[<span style="color:Red">2</span>]__ S. Dieleman, K. W. Willett, J. Dambre, "Rotation-invariant convolutional neural networks for galaxy morphology prediction," *Mon. Not. R. Astron. Soc.* **450**, 1441 (2015). [doi:10.1093/mnras/stv632](https://doi.org/10.1093/mnras/stv632), [arXiv:1503.07077](https://arxiv.org/abs/1503.07077)

__[<span style="color:Red">3</span>]__ C. Lintott *et al.*, "Galaxy Zoo 1: data release of morphological classifications for nearly 900,000 galaxies," *Mon. Not. R. Astron. Soc.* **410**, 166 (2011). [doi:10.1111/j.1365-2966.2010.17432.x](https://doi.org/10.1111/j.1365-2966.2010.17432.x), [arXiv:1007.3265](https://arxiv.org/abs/1007.3265)

__[<span style="color:Red">4</span>]__ D. G. York *et al.*, "The Sloan Digital Sky Survey: Technical Summary," *Astron. J.* **120**, 1579 (2000). [doi:10.1086/301513](https://doi.org/10.1086/301513), [arXiv:astro-ph/0006396](https://arxiv.org/abs/astro-ph/0006396)

__[<span style="color:Red">5</span>]__ Galaxy Zoo – The Galaxy Challenge, Kaggle competition (2013–2014). https://www.kaggle.com/competitions/galaxy-zoo-the-galaxy-challenge

## <span style="color:Orange">Acknowledgements</span>

* Initial version: Mark Neubauer
* Revised: 2026

© Copyright 2026